In [1]:
from pathlib import Path
import pandas as pd

file = Path('../Data/train_data.csv')
df = pd.read_csv(file)

print(df)

        event_id  time_to_tca  mission_id       risk  max_risk_estimate  \
0              0     1.566798           5 -10.204955          -7.834756   
1              0     1.207494           5 -10.355758          -7.848937   
2              0     0.952193           5 -10.345631          -7.847406   
3              0     0.579669           5 -10.337809          -7.845880   
4              0     0.257806           5 -10.391260          -7.852942   
...          ...          ...         ...        ...                ...   
162629     13153     3.029751           1  -7.108630          -5.142668   
162630     13153     2.799253           1  -7.070070          -5.137869   
162631     13153     2.385399           1  -7.066209          -5.137034   
162632     13153     2.043339           1  -7.028307          -5.131297   
162633     13153     1.618095           1  -7.172372          -5.152181   

        max_risk_scaling  miss_distance  relative_speed  relative_position_r  \
0               8.6

In [2]:
# Each event's earliest update, and its whole final update
earliest_update = df.groupby('event_id')['time_to_tca'].max()

final_update = df.loc[
    df.groupby('event_id')['time_to_tca'].idxmin()
].set_index('event_id')

In [3]:
# Keep events with data from 2+ days out
valid_events = earliest_update >= 2
print(valid_events.sum())

11942


In [4]:
# Label: did the event end dangerous (final risk of -6 or higher)?
y = final_update['risk'] >= -6
y = y[valid_events]
print(y.sum())
print(y.mean())

323
0.02704739574610618


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score

# Split the events: 80% to learn from, 20% held back for the final score
train_ids, test_ids = train_test_split(
    y.index, test_size=0.2, stratify=y, random_state=42
)

print(len(train_ids), len(test_ids))
print(y.loc[train_ids].sum(), y.loc[train_ids].mean())
print(y.loc[test_ids].sum(), y.loc[test_ids].mean())



9553 2389
258 0.02700722286192819
65 0.027208036835496024


In [6]:
# Baseline forecast: each event's latest risk from 2+ days out
observed = df[df['time_to_tca'] >= 2]
latest = observed.loc[
    observed.groupby('event_id')['time_to_tca'].idxmin()
].set_index('event_id')

# Score it only on the test events
y_test = y.loc[test_ids]
baseline_pred = latest.loc[test_ids, 'risk'] >= -6

print('Caught:', (baseline_pred & y_test).sum())
print('Missed:', (~baseline_pred & y_test).sum())
print('False alarms:', (baseline_pred & ~y_test).sum())
print('Baseline F2:', fbeta_score(y_test, baseline_pred, beta=2))

Caught: 60
Missed: 5
False alarms: 55
Baseline F2: 0.8


In [7]:
# Features: every column of each event's latest update from 2+ days out
X = latest.loc[y.index].drop(columns=['mission_id'])

# c_object_type is text (DEBRIS, PAYLOAD...), so mark it as a category
X['c_object_type'] = X['c_object_type'].astype('category')

# Split the features the same way as the labels
X_train, X_test = X.loc[train_ids], X.loc[test_ids]
y_train = y.loc[train_ids]

print(X.shape)


(11942, 101)


In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold

model = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)

# Each training event gets a probability from a model that didn't train on it
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_probs = cross_val_predict(model, X_train, y_train, cv=folds, method='predict_proba')[:, 1]
train_probs = pd.Series(train_probs, index=X_train.index)

# Average probability for events that were really safe vs really dangerous
print(train_probs.groupby(y_train).describe())


        count      mean       std           min       25%       50%       75%  \
risk                                                                            
False  9295.0  0.008162  0.077320  3.470595e-07  0.000025  0.000031  0.000050   
True    258.0  0.778194  0.363582  2.614698e-05  0.701468  0.995797  0.999483   

            max  
risk             
False  0.999902  
True   0.999950  


In [9]:
import numpy as np

# Try every cutoff and keep the one with the best F2 on the training events
best_cutoff, best_f2 = 0, 0
for cutoff in np.linspace(0.01, 0.99, 99):
    f2 = fbeta_score(y_train, train_probs >= cutoff, beta=2)
    if f2 > best_f2:
        best_cutoff, best_f2 = cutoff, f2

print('Best cutoff:', best_cutoff)
print('F2 on training events:', best_f2)


Best cutoff: 0.2
F2 on training events: 0.8141263940520446


In [10]:
# Train on every training event, then predict the held-back test events
model.fit(X_train, y_train)
test_probs = model.predict_proba(X_test)[:, 1]
model_pred = test_probs >= best_cutoff

print('             Model  Baseline')
print('Caught:      ', (model_pred & y_test).sum(), '   ', (baseline_pred & y_test).sum())
print('Missed:      ', (~model_pred & y_test).sum(), '    ', (~baseline_pred & y_test).sum())
print('False alarms:', (model_pred & ~y_test).sum(), '   ', (baseline_pred & ~y_test).sum())
print('F2:           %.3f   %.3f' % (fbeta_score(y_test, model_pred, beta=2),
                                     fbeta_score(y_test, baseline_pred, beta=2)))


             Model  Baseline
Caught:       56     60
Missed:       9      5
False alarms: 25     55
F2:           0.821   0.800


In [11]:
# Updates from 2+ days out, sorted oldest first within each event
obs = observed.sort_values(['event_id', 'time_to_tca'], ascending=[True, False])

# How much risk and miss distance changed since the previous update
obs['risk_change'] = obs.groupby('event_id')['risk'].diff()
obs['miss_distance_change'] = obs.groupby('event_id')['miss_distance'].diff()

# Summarize each event's history as one row
last_rows = obs.groupby('event_id').tail(1).set_index('event_id')
by_event = obs.groupby('event_id')

trend = pd.DataFrame({
    'risk_change': last_rows['risk_change'],
    'miss_distance_change': last_rows['miss_distance_change'],
    'risk_max': by_event['risk'].max(),
    'risk_min': by_event['risk'].min(),
    'n_updates': by_event.size(),
})

# Add the trend columns to the original features
X2 = X.join(trend)
print(X2.shape)
print(trend.loc[y.index].groupby(y).median())


(11942, 106)
       risk_change  miss_distance_change  risk_max   risk_min  n_updates
risk                                                                    
False      0.00000                   8.0 -8.821599 -30.000000       11.0
True       0.00296                  -4.0 -5.431212  -5.778325        4.0


In [12]:
# Events where the risk can still change: the final update comes after the 2-day mark
can_change = final_update.loc[y.index, 'time_to_tca'] < 2

def compare(features):
    """Model vs baseline F2 on 5 different splits: all test events, and the ones where risk can still change."""
    rows = []
    for seed in range(5):
        tr, te = train_test_split(y.index, test_size=0.2, stratify=y, random_state=seed)

        # Choose the cutoff using training events only
        m = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        probs = cross_val_predict(m, features.loc[tr], y.loc[tr], cv=folds, method='predict_proba')[:, 1]
        cutoffs = np.linspace(0.01, 0.99, 99)
        cutoff = cutoffs[np.argmax([fbeta_score(y.loc[tr], probs >= c, beta=2) for c in cutoffs])]

        # Train on all training events, then predict the test events
        m.fit(features.loc[tr], y.loc[tr])
        model_pred = pd.Series(m.predict_proba(features.loc[te])[:, 1] >= cutoff, index=te)
        base_pred = latest.loc[te, 'risk'] >= -6
        y_te = y.loc[te]
        hard = can_change.loc[te]

        rows.append({
            'model (all)': fbeta_score(y_te, model_pred, beta=2),
            'baseline (all)': fbeta_score(y_te, base_pred, beta=2),
            'model (can change)': fbeta_score(y_te[hard], model_pred[hard], beta=2),
            'baseline (can change)': fbeta_score(y_te[hard], base_pred[hard], beta=2),
            'model misses (can change)': (~model_pred[hard] & y_te[hard]).sum(),
            'baseline misses (can change)': (~base_pred[hard] & y_te[hard]).sum(),
            'model false alarms (can change)': (model_pred[hard] & ~y_te[hard]).sum(),
            'baseline false alarms (can change)': (base_pred[hard] & ~y_te[hard]).sum(),
        })
    return pd.DataFrame(rows)

print(compare(X).mean())
print(compare(X2).mean())


model (all)                            0.797719
baseline (all)                         0.816990
model (can change)                     0.453769
baseline (can change)                  0.498666
model misses (can change)              7.800000
baseline misses (can change)           4.800000
model false alarms (can change)       29.000000
baseline false alarms (can change)    48.200000
dtype: float64


model (all)                            0.812136
baseline (all)                         0.816990
model (can change)                     0.502087
baseline (can change)                  0.498666
model misses (can change)              6.400000
baseline misses (can change)           4.800000
model false alarms (can change)       32.000000
baseline false alarms (can change)    48.200000
dtype: float64


In [13]:
# For the README: how the baseline does on each group of events
baseline_all = latest.loc[y.index, 'risk'] >= -6
for name, group in [('No updates after the 2-day mark', ~can_change), ('Risk can still change', can_change)]:
    yy, pp = y[group], baseline_all[group]
    print(f'{name}: {group.sum()} events, {yy.sum()} dangerous, caught {(pp & yy).sum()}, '
          f'missed {(~pp & yy).sum()}, false alarms {(pp & ~yy).sum()}, F2 {fbeta_score(yy, pp, beta=2):.2f}')

No updates after the 2-day mark: 3050 events, 230 dangerous, caught 230, missed 0, false alarms 0, F2 1.00
Risk can still change: 8892 events, 93 dangerous, caught 70, missed 23, false alarms 269, F2 0.49
